In [1]:
import os

# If notebook is inside src/, move up one directory to project root
if os.getcwd().endswith("src"):
    os.chdir("..")

print("Working directory:", os.getcwd())



Working directory: /Users/biotechiestefnie/Desktop/Algorithms_Final_Project


# Prototype Run of Markov Model for Positive and Negative Datasets of Each Structural Class

## 1. Add src/ to Path and Import all Packages/Modules

In [2]:
import sys
sys.path.append("src")

from data_loading import load_class_seqs, read_fasta
from train_models import train_all_models
from classify import (
    classify_test_set,
    compute_accuracy,
    compute_summary_stats,
    confusion_matrix,
    evaluate_model_performance
)
from reporting import write_all_outputs




## 2. Define prototype FASTA paths

In [3]:
class_fasta_paths = {
    "promoter_positive": "data/prototype/promoters_positive.fa",
    "promoter_negative": "data/prototype/promoters_negative.fa",
    "exon_positive":     "data/prototype/exons_positive.fa",
    "exon_negative":     "data/prototype/exons_negative.fa",
    "intron_positive":   "data/prototype/introns_positive.fa",
    "intron_negative":   "data/prototype/introns_negative.fa",
    "repeat_positive":   "data/prototype/repeats_positive.fa",
    "repeat_negative":   "data/prototype/repeats_negative.fa"
}


### Debugging Cell- Execution Freezing at Step 3 Infinitely Running with No Error

In [4]:
import os

for label, path in class_fasta_paths.items():
    print(label, "→ exists:", os.path.exists(path))
    if os.path.exists(path):
        print("   size:", os.path.getsize(path))


promoter_positive → exists: True
   size: 160200
promoter_negative → exists: True
   size: 160200
exon_positive → exists: True
   size: 344945
exon_negative → exists: True
   size: 376810
intron_positive → exists: True
   size: 2442694
intron_negative → exists: True
   size: 2758081
repeat_positive → exists: True
   size: 55224
repeat_negative → exists: True
   size: 66385


## 3. Load Prototype Sequences into Model for all Classes

In [5]:
training_data = load_class_seqs(class_fasta_paths)
len(training_data)



8

## 4. Prepare Test Sequences and True Labels

In [6]:
# Build test set and true label mapping for prototype evaluation

test_sequences = []  # Initialize list to hold test seqs
true_labels = {}  # Initialize dict mapping seq -> true class

# Extract every seq for every class
for class_label, seq_list in training_data.items():
    for rec in seq_list:
        seq_str = rec
        test_sequences.append(seq_str)
        true_labels[seq_str] = class_label

len(test_sequences), len(true_labels)


(2400, 2400)

In [7]:
for label, seqs in training_data.items():
    print(label, type(seqs[0]))


promoter_positive <class 'str'>
promoter_negative <class 'str'>
exon_positive <class 'str'>
exon_negative <class 'str'>
intron_positive <class 'str'>
intron_negative <class 'str'>
repeat_positive <class 'str'>
repeat_negative <class 'str'>


## 5. Train Markov Models for all Classes and all k Values

In [8]:
# Train Markov models for all prototype classes over all orders k
# Parameter values
k_values = [1, 2, 3]  # k values under evaluation
alpha = 1  # Laplace smoothing constant

models = train_all_models(training_data, k_values, alpha)

models


{1: {'promoter_positive': {'T': {'A': 0.1284616697968243,
    'C': 0.30302202492743724,
    'G': 0.32415912583233736,
    'T': 0.24435717944340105},
   'C': {'A': 0.18119066234781636,
    'C': 0.33311739081872,
    'G': 0.24950295990170893,
    'T': 0.23616664805093265},
   'A': {'A': 0.23565528835611432,
    'C': 0.22308310795994446,
    'G': 0.38458446020027776,
    'T': 0.15667714348366346},
   'G': {'A': 0.18718629443730037,
    'C': 0.3085825693781889,
    'G': 0.3537146886796366,
    'T': 0.1505164475048741},
   'N': {'A': 0.0025380710659898475,
    'C': 0.0025380710659898475,
    'G': 0.005076142131979695,
    'T': 0.0025380710659898475}},
  'promoter_negative': {'G': {'A': 0.17466872110939907,
    'C': 0.3068104776579353,
    'G': 0.33294298921417564,
    'T': 0.1836466358500257},
   'A': {'A': 0.1969619326500732,
    'C': 0.28784773060029284,
    'G': 0.3152635431918009,
    'T': 0.19816983894582724},
   'T': {'A': 0.19031214848143982,
    'C': 0.29123312710911137,
    'G': 0.

## 6. Classify Each Sequence Against all Classes for Every Order k

In [9]:
# Classify all prototype sequences for each k value

all_classification_results = {}  # Initialize dictionary for results

for k in k_values:
    # classify_test_set() returns: (sequence, predicted_class, score)
    results_k = classify_test_set(test_sequences, models, k)
    all_classification_results[k] = results_k

all_classification_results


{1: [('TCACTGCAACCTCCGCCTCCTGGGTTCAAGTGATTTCCAGCTAATTTTTGTATTTTTAGTAGAGATGGGGTTTCACCATGTTGGCCAGGCTGGTCTTGAACTCCTGACCTCAAGTGATCCGCCCACTTCGGCCTCCCAAAGTGCTAGGATTACAGACATGAGTCACCACACCCAGACCCCAAAATAGGATTTTCTTAAAGAGCGCTCAGCTTAATTAAAAGTGGATATCTGGGGGGCTGGCACGCGGCAGCGTTGCGGGTGGGAGCGGCTGCAACTCTGGTGCCTGAGGAGCGATACCAAGAGAAATCATCACCCACAATTGGGCCAGTGCAGCAATCAGATTGGGTTCGAGTTCTGGAAACAGCTGTGCACTGAGCATGGTATCAGCCCCAAGGGCACCATGGAGGAGTTCGCTACTGAGGGCACTGACCACAAGGACATCTTTTTCTACCAGGCAGACGATGAGCACTACATCCCCCGGGCTGTGCTGCTGGACCTGGA',
   None,
   -inf),
  ('CACCTCAGGAAAAGGAGGAACATTAGCGCTGGTACAGCCGCCAAGACTTCCGGAAGCTCCATCAATCACTTAGCTCCCCTGGTCTCTCGGGTAGGCATCCGCCCACCGCCCCCATCGTCACTTCCGTCGGCCGACAGCACCCAAGATTGACAGGCGCGGACGTCCAATCAGATGCGGGCCCAGCCCCAAAGCCGAGACGAGGGGCGGGTTTGAGAGCGGAAAGCCCCACCCCTTGCCTGAGTGTGACGTCAGAATCACCATGGCCAGCTATCCTTACCGGCAGGTGAGTGTGTGAGGGGCCCGCGAATCCAGGTAGCGGCGGTGCCAGGCGCAGGCCCGACGTCCCCCTGCTCTTTCTCCCCGCTTTCTCCGCGCCCTTCCCACGATGGGTTCGCTTCAGCGAGGCCTTGCGCTGTAGAGAACCGAAAAGGGACCGTCGGTCGCGCCCTGCTCACGTCATGAAGGA

## 7. Build Data Structure for Evaluation

In [10]:
# Build full all_results structure required by evaluate_model_performance()

all_results = {}

for k in k_values:
    results_k = []
    raw = all_classification_results[k]   # from Cell 6

    for (seq, pred, score) in raw:
        # Placeholder class_scores: same score for all classes
        class_scores = {cls: score for cls in models[k].keys()}

        # Append full 4-tuple
        results_k.append((seq, pred, score, class_scores))

    all_results[k] = results_k

all_results



{1: [('TCACTGCAACCTCCGCCTCCTGGGTTCAAGTGATTTCCAGCTAATTTTTGTATTTTTAGTAGAGATGGGGTTTCACCATGTTGGCCAGGCTGGTCTTGAACTCCTGACCTCAAGTGATCCGCCCACTTCGGCCTCCCAAAGTGCTAGGATTACAGACATGAGTCACCACACCCAGACCCCAAAATAGGATTTTCTTAAAGAGCGCTCAGCTTAATTAAAAGTGGATATCTGGGGGGCTGGCACGCGGCAGCGTTGCGGGTGGGAGCGGCTGCAACTCTGGTGCCTGAGGAGCGATACCAAGAGAAATCATCACCCACAATTGGGCCAGTGCAGCAATCAGATTGGGTTCGAGTTCTGGAAACAGCTGTGCACTGAGCATGGTATCAGCCCCAAGGGCACCATGGAGGAGTTCGCTACTGAGGGCACTGACCACAAGGACATCTTTTTCTACCAGGCAGACGATGAGCACTACATCCCCCGGGCTGTGCTGCTGGACCTGGA',
   None,
   -inf,
   {'promoter_positive': -inf,
    'promoter_negative': -inf,
    'exon_positive': -inf,
    'exon_negative': -inf,
    'intron_positive': -inf,
    'intron_negative': -inf,
    'repeat_positive': -inf,
    'repeat_negative': -inf}),
  ('CACCTCAGGAAAAGGAGGAACATTAGCGCTGGTACAGCCGCCAAGACTTCCGGAAGCTCCATCAATCACTTAGCTCCCCTGGTCTCTCGGGTAGGCATCCGCCCACCGCCCCCATCGTCACTTCCGTCGGCCGACAGCACCCAAGATTGACAGGCGCGGACGTCCAATCAGATGCGGGCCCAGCCCCAAAGCCGAGACGAGGGGCGGGTTTGAGAGCGGAAAGCCCCACCCCT

## 8a. Full Classification Analysis for a Single Order k

In [11]:
k = 2

raw_classification_k = all_classification_results[k]

# 4-tuples for this k
results_k = all_results[k]

# Convert to dicts for per-k functions
results_dicts_k = [
    {
        "sequence": seq,
        "predicted_class": pred,
        "log_likelihood": score,
        "true_class": true_labels[seq]
    }
    for (seq, pred, score, _) in results_k
]

# Summary statistics (single k)
summary_stats_k = compute_summary_stats(results_dicts_k)

# Per-k accuracy
accuracy_k = compute_accuracy(results_dicts_k)

# Per-k confusion matrix
confusion_matrix_k = confusion_matrix(results_dicts_k)

#  Full cross-k metrics, restricted to single order k, including accuracy_vs_k, confusion_matrices, likelihood_distributions for k=2)
evaluation_k = evaluate_model_performance(
    {k: results_k},
    true_labels
)

summary_stats_k, accuracy_k, confusion_matrix_k, evaluation_k

({'total': 2400, 'unclassified': 2400, 'neg_inf': 2400, 'class_counts': {}},
 0.0,
 {'exon_negative': {'exon_negative': 0,
   'exon_positive': 0,
   'intron_negative': 0,
   'intron_positive': 0,
   'promoter_negative': 0,
   'promoter_positive': 0,
   'repeat_negative': 0,
   'repeat_positive': 0},
  'exon_positive': {'exon_negative': 0,
   'exon_positive': 0,
   'intron_negative': 0,
   'intron_positive': 0,
   'promoter_negative': 0,
   'promoter_positive': 0,
   'repeat_negative': 0,
   'repeat_positive': 0},
  'intron_negative': {'exon_negative': 0,
   'exon_positive': 0,
   'intron_negative': 0,
   'intron_positive': 0,
   'promoter_negative': 0,
   'promoter_positive': 0,
   'repeat_negative': 0,
   'repeat_positive': 0},
  'intron_positive': {'exon_negative': 0,
   'exon_positive': 0,
   'intron_negative': 0,
   'intron_positive': 0,
   'promoter_negative': 0,
   'promoter_positive': 0,
   'repeat_negative': 0,
   'repeat_positive': 0},
  'promoter_negative': {'exon_negative': 

## 8b. Compute Summary Statistics for all Orders k

In [12]:
# Classifications per test seq for all orders k
raw_classification_all_k = all_classification_results

# Convert classify results for test seqs into dicts for all orders k
results_dicts_all_k = {
    k: [
        {
            "sequence": seq,
            "predicted_class": pred,
            "log_likelihood": score,
            "true_class": true_labels[seq]
        }
        for (seq, pred, score) in raw_classification_all_k[k]
    ]
    for k in k_values
}

# Summary statistics for all orders k
summary_stats_all_k = {
    k: compute_summary_stats(results_dicts_all_k[k])
    for k in k_values
}

# Accuracy for all orders k
accuracy_all_k = {
    k: compute_accuracy(results_dicts_all_k[k])
    for k in k_values
}

# Confusion matrices for all orders k
confusion_matrices_all_k = {
    k: confusion_matrix(results_dicts_all_k[k])
    for k in k_values
}

# Full cross-k evaluation metrics
# incl accuracy_vs_k, cross-k confusion matrices, likelihood distributions
evaluation_all_k = evaluate_model_performance(
    all_results,
    true_labels
)

# Return analysis results for all k
raw_classification_all_k, summary_stats_all_k, accuracy_all_k, confusion_matrices_all_k, evaluation_all_k


({1: [('TCACTGCAACCTCCGCCTCCTGGGTTCAAGTGATTTCCAGCTAATTTTTGTATTTTTAGTAGAGATGGGGTTTCACCATGTTGGCCAGGCTGGTCTTGAACTCCTGACCTCAAGTGATCCGCCCACTTCGGCCTCCCAAAGTGCTAGGATTACAGACATGAGTCACCACACCCAGACCCCAAAATAGGATTTTCTTAAAGAGCGCTCAGCTTAATTAAAAGTGGATATCTGGGGGGCTGGCACGCGGCAGCGTTGCGGGTGGGAGCGGCTGCAACTCTGGTGCCTGAGGAGCGATACCAAGAGAAATCATCACCCACAATTGGGCCAGTGCAGCAATCAGATTGGGTTCGAGTTCTGGAAACAGCTGTGCACTGAGCATGGTATCAGCCCCAAGGGCACCATGGAGGAGTTCGCTACTGAGGGCACTGACCACAAGGACATCTTTTTCTACCAGGCAGACGATGAGCACTACATCCCCCGGGCTGTGCTGCTGGACCTGGA',
    None,
    -inf),
   ('CACCTCAGGAAAAGGAGGAACATTAGCGCTGGTACAGCCGCCAAGACTTCCGGAAGCTCCATCAATCACTTAGCTCCCCTGGTCTCTCGGGTAGGCATCCGCCCACCGCCCCCATCGTCACTTCCGTCGGCCGACAGCACCCAAGATTGACAGGCGCGGACGTCCAATCAGATGCGGGCCCAGCCCCAAAGCCGAGACGAGGGGCGGGTTTGAGAGCGGAAAGCCCCACCCCTTGCCTGAGTGTGACGTCAGAATCACCATGGCCAGCTATCCTTACCGGCAGGTGAGTGTGTGAGGGGCCCGCGAATCCAGGTAGCGGCGGTGCCAGGCGCAGGCCCGACGTCCCCCTGCTCTTTCTCCCCGCTTTCTCCGCGCCCTTCCCACGATGGGTTCGCTTCAGCGAGGCCTTGCGCTGTAGAGAACCGAAAAGGGACCGTCGGTCGCGCCCTGCTCACGTCATGA

## 9. Report Results for Single k and all Orders k to Output Datafiles

In [13]:
run = "prototype"        # this is the run name (prototype run)
base_dir = "results"

# write outputs for single k (k = 2)
run_name_single = "prototype_k2"

write_all_outputs(
    results=all_results[2],          # 4-tuples for k=2
    stats=summary_stats_k,           # summary stats for k=2
    metrics=evaluation_k,            # evaluation metrics for k=2
    dataset=run,                     # run folder name
    run_name=run_name_single,
    base_dir=base_dir
)

# write outputs for all k
run_name_all = "prototype_all_k"

# flatten all 4-tuples across k
all_results_flat = []
for k in k_values:
    all_results_flat.extend(all_results[k])

# build metrics dict in the structure write_all_outputs expects
combined_metrics_all_k = {
    "accuracy_vs_k": evaluation_all_k["accuracy_vs_k"],
    "confusion_matrices": evaluation_all_k["confusion_matrices"],
    "likelihood_distributions": evaluation_all_k["likelihood_distributions"]
}

# ---- FIX: DO NOT PASS summary_stats_all_k INTO write_all_outputs ----
# Instead, pass a dummy stats dict that satisfies write_summary_stats()
# but will be ignored because we skip summary stats for all-k.

dummy_stats = {
    "total": 0,
    "unclassified": 0,
    "neg_inf": 0,
    "class_counts": {}
}

write_all_outputs(
    results=all_results_flat,
    stats=dummy_stats,               # <-- FIXED
    metrics=combined_metrics_all_k,
    dataset=run,
    run_name=run_name_all,
    base_dir=base_dir
)

